# SpecDist — Adobe AI Platform (AIP)

**Sensei FS persistent storage · 4-hour guaranteed session · up to 3 GPUs/user**

| Cell | What it does | Time |
|------|-------------|------|
| 0. Bootstrap | One-time: clone → deps → auth → run | ~3 min + model dl (once) |
| 1. Resume | After 4-hour timeout — pipeline skips completed steps | ~1 min |
| 2. Monitor | State + log tail (auto-refresh option) | instant |
| 3. Verify | Check artifacts before session ends | instant |
| 4. Parallel | All losses simultaneously (multi-GPU or CPU) | ~2-3 h |

---

### Before you start

**1. Set secrets** — in the AIP VS Code terminal *before* running this notebook:
```bash
export GITHUB_TOKEN="ghp_..."        # PAT (repo scope) — only if repo is private
export WANDB_API_KEY="..."           # https://wandb.ai/authorize
export HF_TOKEN="hf_..."             # optional — Qwen3 models are public
```
Add to `~/.bashrc` for persistence across sessions so you don't have to re-export.

**2. Check your Sensei FS path** — the default storage uses your LDAP (`$USER`):
```bash
echo /sensei-fs-3/users/$USER        # verify this path exists
ls /sensei-fs-3/users/$USER          # check what's already there
```
Override `STORAGE_ROOT` in Cell 0 if your Sensei FS mount is at a different path.

**3. Verify GPU allocation**: `nvidia-smi` in the terminal.

---

### CONFIG options

| CONFIG | Teacher | Steps | VRAM | Session time | When to use |
|--------|---------|-------|------|-------------|-------------|
| `a100` | Qwen3-8B (BF16) | 2000 | ~19 GB | **~2-3 h** (all losses) | **A100 / V100 32 GB** — fits in one 4-hour session |
| `kaggle` | Qwen3-8B (4-bit NF4) | 1000 | ~7.8 GB | ~5-8 h/loss | T4 / V100 16 GB — multi-session |
| `colab` | Qwen3-4B (BF16) | 500 | ~10.7 GB | ~4 h/loss | T4 safe fallback (no quantization) |
| `colab_lite` | Qwen3-1.7B (BF16) | 300 | ~5.6 GB | ~25 min | Quick smoke test |

**Quick GPU → config decision:**
- `nvidia-smi` shows 40 GB → use `a100` (entire run in one 4-hour session)
- `nvidia-smi` shows 32 GB → use `a100` (same — A100 config targets ≥ 19 GB)
- `nvidia-smi` shows 16 GB → use `kaggle` (8B NF4, multi-session over 4-hour slots)
- Unsure → run `SMOKE = True` first to verify the code path

### 4-hour session limit

AIP interactive sessions guarantee **4 hours** (not unlimited like Lightning AI).  
Always run with `BACKGROUND = True` and use **Cell 1 (Resume)** on the next session.  
The pipeline auto-skips completed steps — no work is lost on session expiry.

**Multi-session strategy for `a100` (A100):**  
12 losses × ~10-17 min each = ~2-3 h total → completes in **one** 4-hour session.  

**Multi-session strategy for `kaggle` config (T4/V100 16 GB):**  
Each loss takes ~5-8 h → requires multiple sessions. Use Cell 1 each time.

### Persistent storage

All artifacts (checkpoints, `results.db`, logs, HF model cache) go to Sensei FS  
and **persist across sessions**. Models download once (~10 min for 8B), then  
load instantly every session. No manual save step required.

### Multi-GPU

If your AIP session has multiple GPUs, `device_map="auto"` (used by all configs)  
automatically shards the teacher model across all available GPUs — no code changes.  
The draft model always trains on `cuda:0`.

**Cell 4 — Parallel training across all losses:**  
Assigns each loss group to a separate GPU (or CPU for GPT-2).  
3 GPUs x 5 losses each = all 15 losses done in ~2-3 h instead of 30-40 h sequential.  
Set `FAMILY = "gpt2"` to run CPU convergence runs alongside Qwen GPU runs.

### W&B

Set `WANDB_API_KEY` to use your personal W&B account at https://wandb.ai.  
If Adobe has an enterprise W&B instance, replace the `WANDB_BASE_URL` env var:  
`export WANDB_BASE_URL=https://adobesensei.wandb.io`  
Leave unset to use the public wandb.ai endpoint.

In [ ]:
# =============================================================================
# Cell 0 — BOOTSTRAP  (run once per fresh AIP session)
#
# Set secrets in the VS Code terminal BEFORE running this cell:
#   export GITHUB_TOKEN="ghp_..."      # required for private repo
#   export WANDB_API_KEY="..."         # from https://wandb.ai/authorize
#   export HF_TOKEN="hf_..."           # optional (Qwen3 models are public)
#
# After a session timeout use Cell 1 (Resume) instead — it is self-contained.
# =============================================================================

import os

# -- Edit these ---------------------------------------------------------------
REPO_URL     = "https://github.com/Rmuk655/Distill-Spec-Research.git"

# Sensei FS: persistent across sessions. Uses $USER (your LDAP) automatically.
# Override if your Sensei FS path is different, e.g. a shared tenant dir.
_LDAP        = os.environ.get("USER", "YOUR_LDAP_HERE")
SENSEI_ROOT  = f"/sensei-fs-3/users/{_LDAP}"
REPO_DIR     = f"{SENSEI_ROOT}/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = f"{SENSEI_ROOT}/specdist"   # checkpoints + results.db + logs

# CONFIG — choose based on your GPU (run nvidia-smi in terminal to check VRAM):
#   a100        — A100 / ≥ 30 GB VRAM: 8B teacher BF16, 2000 steps, full run ~2-3 h
#   kaggle      — T4 / V100 16 GB:      8B teacher NF4,  1000 steps, ~5-8 h/loss
#   colab       — T4 safe fallback:     4B teacher BF16, 500 steps,  ~4 h/loss
#   colab_lite  — any GPU, quick test:  1.7B teacher,    300 steps,  ~25 min
CONFIG       = "a100"   # ← change this if your GPU has < 30 GB VRAM

SMOKE        = False   # True = 10-step crash check before full run (run this first!)
BACKGROUND   = True    # ALWAYS True: 4-hour AIP session limit requires background mode
LOSSES       = None    # None = all losses  |  "kl,ebe" = subset for targeted runs
EXTRA_ARGS   = []
# -----------------------------------------------------------------------------

import subprocess, sys

# Verify Sensei FS is accessible
if not os.path.isdir(SENSEI_ROOT):
    print(f"WARNING: Sensei FS path not found: {SENSEI_ROOT}")
    print(f"  Check with: ls /sensei-fs-3/users/")
    print(f"  Set SENSEI_ROOT manually above if your path differs.")
else:
    print(f"Sensei FS: {SENSEI_ROOT} (accessible)")

gh = os.environ.get("GITHUB_TOKEN", "")
if not gh:
    print("WARNING: GITHUB_TOKEN not set — clone will fail for a private repo.")
    print("  export GITHUB_TOKEN='ghp_...' in the VS Code terminal, then re-run.")

clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.strip())
        raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned to Sensei FS")
else:
    if gh:
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url],
                       capture_output=True)
    r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                       capture_output=True, text=True)
    print(r.stdout.strip() or "Already up to date")
    if r.returncode != 0:
        print("[pull error]", r.stderr.strip())

sys.modules.pop("deploy_utils", None)
sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import bootstrap, run_pipeline

# AIP-specific: HF model cache → Sensei FS (persists across sessions).
# First session: 8B model downloads to Sensei FS (~10 min).
# Every subsequent session: loads from persistent cache instantly.
hf_cache = os.path.join(STORAGE_ROOT, "hf_cache")
os.makedirs(hf_cache, exist_ok=True)
os.environ["HF_HOME"]            = hf_cache
os.environ["TRANSFORMERS_CACHE"] = hf_cache
for _flag in ("TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE", "HF_HUB_OFFLINE"):
    os.environ.pop(_flag, None)
print(f"HF cache  : {hf_cache}")

# W&B: set WANDB_API_KEY in terminal. Set WANDB_BASE_URL for an enterprise instance.
# e.g. export WANDB_BASE_URL=https://adobesensei.wandb.io

# bootstrap() installs deps, authenticates W&B + HF, downloads training data.
# storage_root → Sensei FS (persistent): no mount_drive, no restore_checkpoints.
bootstrap(CONFIG, STORAGE_ROOT, REPO_DIR, gbv_dir=GBV_DIR,
          warn_vram_below_gb=12.0)

_ = run_pipeline(CONFIG, STORAGE_ROOT, GBV_DIR,
                 smoke=SMOKE, losses=LOSSES,
                 background=BACKGROUND, extra_args=EXTRA_ARGS)

In [ ]:
# =============================================================================
# Cell 1 — RESUME  (after 4-hour session expiry or manual restart)
#
# Self-contained: re-clones if needed, re-installs deps, then resumes.
# The pipeline reads pipeline_state_*.json and skips completed steps.
# Sensei FS artifacts (checkpoints, results.db, HF cache) are already there.
#
# If secrets expired, re-export them in the terminal before running:
#   export WANDB_API_KEY="..."  &&  export GITHUB_TOKEN="ghp_..."
# =============================================================================

import os, subprocess, sys

# -- Edit these (must match Cell 0) -------------------------------------------
REPO_URL     = "https://github.com/Rmuk655/Distill-Spec-Research.git"
_LDAP        = os.environ.get("USER", "YOUR_LDAP_HERE")
SENSEI_ROOT  = f"/sensei-fs-3/users/{_LDAP}"
REPO_DIR     = f"{SENSEI_ROOT}/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = f"{SENSEI_ROOT}/specdist"
CONFIG       = "a100"   # must match Cell 0
# -----------------------------------------------------------------------------

gh = os.environ.get("GITHUB_TOKEN", "")
clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.strip())
        raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned")
else:
    if gh:
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url],
                       capture_output=True)
    r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                       capture_output=True, text=True)
    print(r.stdout.strip() or "Already up to date")
    if r.returncode != 0:
        print("[pull error]", r.stderr.strip())

# Restore HF cache env var — Sensei FS cache is already populated
hf_cache = os.path.join(STORAGE_ROOT, "hf_cache")
os.environ["HF_HOME"]            = hf_cache
os.environ["TRANSFORMERS_CACHE"] = hf_cache
for _flag in ("TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE", "HF_HUB_OFFLINE"):
    os.environ.pop(_flag, None)

sys.modules.pop("deploy_utils", None)
sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import bootstrap, run_pipeline

# Re-install deps (wiped when pod restarts), re-auth W&B + HF.
# No model download: HF cache is already on Sensei FS.
bootstrap(CONFIG, STORAGE_ROOT, REPO_DIR, gbv_dir=GBV_DIR,
          warn_vram_below_gb=12.0)

print(f"\nResuming {CONFIG} — completed steps are skipped automatically.\n")
_ = run_pipeline(CONFIG, STORAGE_ROOT, GBV_DIR, background=True)

In [ ]:
# =============================================================================
# Cell 2 — MONITOR  (safe to run any time, including while pipeline runs)
# Set AUTO_REFRESH = True for a live tail; interrupt the cell to stop.
# =============================================================================

import os, sys

_LDAP        = os.environ.get("USER", "YOUR_LDAP_HERE")
SENSEI_ROOT  = f"/sensei-fs-3/users/{_LDAP}"
REPO_DIR     = f"{SENSEI_ROOT}/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = f"{SENSEI_ROOT}/specdist"
CONFIG       = "a100"   # must match Cell 0/1
AUTO_REFRESH = False           # True = live tail loop (interrupt cell to stop)
REFRESH_SECS = 20

sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import monitor

monitor(STORAGE_ROOT, CONFIG, auto_refresh=AUTO_REFRESH, refresh_secs=REFRESH_SECS)

In [ ]:
# =============================================================================
# Cell 3 — VERIFY  (check artifacts before session ends / for peace of mind)
#
# Sensei FS is persistent — nothing is lost on session expiry.
# This cell is informational: it shows what has been saved.
# =============================================================================

import os, json, pathlib, shutil

_LDAP        = os.environ.get("USER", "YOUR_LDAP_HERE")
SENSEI_ROOT  = f"/sensei-fs-3/users/{_LDAP}"
STORAGE_ROOT = f"{SENSEI_ROOT}/specdist"

ckpt_dir = os.path.join(STORAGE_ROOT, "checkpoints")
db_path  = os.path.join(STORAGE_ROOT, "results.db")
log_path = os.path.join(STORAGE_ROOT, "logs", "pipeline_output.log")
hf_cache = os.path.join(STORAGE_ROOT, "hf_cache")

print(f"Storage root : {STORAGE_ROOT}")
print()
print(f"results.db   : {os.path.getsize(db_path):,} bytes" if os.path.exists(db_path)
      else "results.db   : NOT FOUND")
print(f"checkpoints/ : {sorted(os.listdir(ckpt_dir)) if os.path.isdir(ckpt_dir) else 'NOT FOUND'}")
print(f"log lines    : {sum(1 for _ in open(log_path))}" if os.path.exists(log_path)
      else "log          : NOT FOUND")

# Sensei FS usage
try:
    total = shutil.disk_usage(SENSEI_ROOT)
    used_gb  = (total.total - total.free) / 1024**3
    free_gb  = total.free / 1024**3
    print(f"\nSensei FS    : {used_gb:.1f} GB used, {free_gb:.1f} GB free")
except Exception as e:
    print(f"Disk usage   : could not compute ({e})")

print()
print("All artifacts are on Sensei FS — they persist across sessions automatically.")
print("No manual save needed. Resume with Cell 1 on next session.")

In [ ]:
# =============================================================================
# Cell 4 — PARALLEL TRAINING  (all losses simultaneously)
#
# On AIP with multiple GPUs (up to 3): assigns each loss group to a separate
# GPU so all 15 losses train simultaneously.
#   3 GPUs x 5 losses = ~2-3 h total  (vs 30-40 h sequential on one GPU)
#
# For GPT-2 (FAMILY="gpt2"): launches all losses as CPU background processes.
# No VRAM needed — useful alongside a Qwen GPU run, or on the ATS Cloud server.
#
# Logs go to STORAGE_ROOT/logs/parallel/<family>_<loss>.log
# Results land in the same results.db — dashboard model-family filter
# (DistilGPT-2, LLaMA, Qwen chips) distinguishes pairs automatically.
# =============================================================================

import os, subprocess, sys

# -- Edit these (must match Cell 0) -------------------------------------------
_LDAP        = os.environ.get("USER", "YOUR_LDAP_HERE")
SENSEI_ROOT  = f"/sensei-fs-3/users/{_LDAP}"
REPO_DIR     = f"{SENSEI_ROOT}/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = f"{SENSEI_ROOT}/specdist"

# FAMILY: which model pair to train
#   "gpt2"  — distilgpt2 (82M) → gpt2-medium (355M)  — CPU, no VRAM
#   "llama" — Llama-3.2-1B → 3B-Instruct              — GPU (needs HF login)
#   "qwen"  — Qwen2.5-0.5B → Qwen3-0.6B/8B            — GPU, use a100/kaggle cfg
FAMILY = "gpt2"     # ← "gpt2" for CPU run; "qwen" for multi-GPU Qwen run

STEPS  = 1000       # steps per loss (gpt2: ~2-4 hr/loss CPU; qwen: ~10-17 min/loss A100)

# GPU assignment for FAMILY="qwen" (ignored for CPU families)
# Adjust group sizes to match your GPU count (AIP gives up to 3)
GPU_GROUPS = {
    0: ["kl", "rev_kl", "jsd", "l1", "kl_tree"],
    1: ["rev_kl_tree", "jsd_tree", "bv_tree", "gbv_tree", "traversal_tree"],
    2: ["naive_tree", "nss_tree", "specinfer_tree", "spectr_tree", "khisti_tree"],
}
# Losses for CPU families (ebe/online excluded per docs/ISSUES.md)
CPU_LOSSES = [
    "kl", "rev_kl", "jsd", "l1",
    "kl_tree", "rev_kl_tree", "jsd_tree",
    "bv_tree", "gbv_tree", "traversal_tree",
    "naive_tree", "nss_tree", "specinfer_tree", "spectr_tree", "khisti_tree",
]
# ---------------------------------------------------------------------------

sys.path.insert(0, GBV_DIR)
from core.model_families import get_family

family_obj = get_family(FAMILY)
draft_id   = family_obj.default_draft_model_id
target_id  = family_obj.default_target_model_id
use_cpu    = FAMILY in ("gpt2",)   # add "llama" here if no GPU

trainer    = os.path.join(GBV_DIR, "algorithms", "distillspec_gbv", "trainer.py")
ckpt_root  = os.path.join(STORAGE_ROOT, "checkpoints")
dataset    = os.path.join(GBV_DIR, "core", "datasets", "raw", "gsm8k_train.jsonl")
log_dir    = os.path.join(STORAGE_ROOT, "logs", "parallel")
os.makedirs(log_dir, exist_ok=True)
os.makedirs(ckpt_root, exist_ok=True)

# Make HF online so models can download if not cached
for _flag in ("TRANSFORMERS_OFFLINE", "HF_HUB_OFFLINE", "HF_DATASETS_OFFLINE"):
    os.environ.pop(_flag, None)

procs = []

if use_cpu:
    # ── CPU path: all losses in parallel ─────────────────────────────────────
    print(f"Launching {len(CPU_LOSSES)} {FAMILY} losses on CPU in parallel...")
    print(f"  draft={draft_id}  target={target_id}  steps={STEPS}")
    for loss in CPU_LOSSES:
        log_path = os.path.join(log_dir, f"{FAMILY}_{loss}.log")
        log_f = open(log_path, "w")
        p = subprocess.Popen(
            [
                sys.executable, trainer,
                "--model_family", FAMILY,
                "--draft",        draft_id,
                "--target",       target_id,
                "--loss",         loss,
                "--steps",        str(STEPS),
                "--device",       "cpu",
                "--no_wandb",
                "--teacher_temp", "1.0",
                "--dataset",      dataset,
                "--output",       os.path.join(ckpt_root, f"{loss}-{FAMILY}"),
            ],
            stdout=log_f, stderr=subprocess.STDOUT, cwd=GBV_DIR,
        )
        procs.append((loss, p, log_f))
        print(f"  [{loss:20s}] PID {p.pid:6d}  log: parallel/{FAMILY}_{loss}.log")

else:
    # ── GPU path: one group per GPU (Qwen / LLaMA with GPU) ──────────────────
    import torch
    n_gpus = torch.cuda.device_count()
    print(f"Detected {n_gpus} GPU(s). Launching loss groups across GPUs...")
    print(f"  draft={draft_id}  target={target_id}  steps={STEPS}")
    for gpu_idx, losses in GPU_GROUPS.items():
        if gpu_idx >= n_gpus:
            print(f"  GPU {gpu_idx}: not available ({n_gpus} GPUs total) — skipping")
            continue
        for loss in losses:
            env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu_idx)}
            log_path = os.path.join(log_dir, f"{FAMILY}_{loss}_gpu{gpu_idx}.log")
            log_f = open(log_path, "w")
            p = subprocess.Popen(
                [
                    sys.executable, trainer,
                    "--model_family", FAMILY,
                    "--draft",        draft_id,
                    "--target",       target_id,
                    "--loss",         loss,
                    "--steps",        str(STEPS),
                    "--device",       "cuda",
                    "--no_wandb",
                    "--dataset",      dataset,
                    "--output",       os.path.join(ckpt_root, f"{loss}-{FAMILY}"),
                ],
                stdout=log_f, stderr=subprocess.STDOUT,
                env=env, cwd=GBV_DIR,
            )
            procs.append((loss, p, log_f))
            print(f"  GPU{gpu_idx} [{loss:20s}] PID {p.pid:6d}")

print(f"\n{len(procs)} jobs running. Waiting (this cell blocks until all finish)...")
print(f"Tail a log: tail -f {os.path.join(log_dir, FAMILY + '_kl.log')}")

failed = []
for loss, p, log_f in procs:
    rc = p.wait()
    log_f.close()
    status = "OK" if rc == 0 else f"FAILED rc={rc}"
    print(f"  [{loss:20s}] {status}")
    if rc != 0:
        failed.append(loss)

if failed:
    print(f"\n{len(failed)} losses failed: {failed}")
    print(f"Tail logs in: {log_dir}")
else:
    print(f"\nAll {len(procs)} losses trained successfully!")
    print("Next: run experiment.py --eval_only or use the eval step in Cell 0/1.")
